In [2]:
import numpy as np
import pandas as pd
from decimal import Decimal
import re

In [3]:
bitcoin_blocks = pd.read_csv("../data/real_bitcoin_blocks_raw.csv")
coinbase_addresses = pd.read_csv("../data/coinbase_addresses_full.csv")

In [4]:
# Formatting the real_outputs column data with regex and decimal conversion

# This regex restores the comma between entries before we hand the string to eval().
def parse_outputs(s):
    fixed = re.sub(r"\}\s*\n\s*\{", "}, {", s)
    return eval(fixed, {"Decimal": Decimal, "__builtins__": {}})

coinbase_addresses["real_outputs_parsed"] = coinbase_addresses["real_outputs"].apply(parse_outputs)
coinbase_addresses["n_addresses"] = coinbase_addresses["real_outputs_parsed"].apply(len)

In [5]:
coinbase_addresses['block_timestamp'] = pd.to_datetime(coinbase_addresses['block_timestamp'])
coinbase_addresses['block_timestamp'].describe()

count                              810909
mean     2016-05-13 12:52:27.924965+00:00
min             2009-01-03 18:15:05+00:00
25%             2012-10-10 22:19:35+00:00
50%             2016-04-02 20:55:40+00:00
75%             2019-12-15 06:03:40+00:00
max             2023-10-06 13:37:21+00:00
Name: block_timestamp, dtype: object

In [6]:
def get_primary_miner(coinbase_payouts):
    if coinbase_payouts == []:
        return (None, 0)
    
    highest_payout = max(coinbase_payouts, key=lambda x: x["value"])
    
    highest_payouts = []
    for payout in coinbase_payouts:
        if payout["value"] == highest_payout["value"]:
            highest_payouts.append(payout)
    
    return (highest_payouts[0]["address"], len(highest_payouts))

In [7]:
coinbase_addresses['primary_miner_result'] = coinbase_addresses['real_outputs_parsed'].apply(get_primary_miner)
coinbase_addresses['primary_miner_address'] = coinbase_addresses['primary_miner_result'].str[0]
coinbase_addresses['primary_miner_ties'] = coinbase_addresses['primary_miner_result'].str[1]
coinbase_addresses

,block_number,block_timestamp,real_outputs,real_outputs_parsed,n_addresses,primary_miner_result,primary_miner_address,primary_miner_ties
0,755688,2022-09-25 21:17:12+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,1,"(18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX, 1)",18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX,1
1,755658,2022-09-25 15:04:25+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,1,"(18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX, 1)",18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX,1
2,756371,2022-09-30 13:14:40+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,1,"(18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX, 1)",18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX,1
3,754843,2022-09-19 20:29:55+00:00,[{'address': '3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8...,[{'address': '3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8...,1,"(3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8TZ, 1)",3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8TZ,1
4,755220,2022-09-22 11:21:06+00:00,[{'address': '3C9sAKXrBVpJVe3b738yik4LPHpPmceB...,[{'address': '3C9sAKXrBVpJVe3b738yik4LPHpPmceB...,1,"(3C9sAKXrBVpJVe3b738yik4LPHpPmceBgd, 1)",3C9sAKXrBVpJVe3b738yik4LPHpPmceBgd,1
...,...,...,...,...,...,...,...,...
810904,762082,2022-11-07 04:51:06+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,1,"(12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL, 1)",12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL,1
810905,761961,2022-11-06 09:03:17+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,1,"(12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL, 1)",12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL,1
810906,763253,2022-11-15 08:27:13+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...,1,"(12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL, 1)",12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8KL,1
810907,762893,2022-11-12 15:41:42+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,1,"(18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX, 1)",18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX,1


In [8]:
# join both dataframes
merged_df = pd.merge(bitcoin_blocks, coinbase_addresses, on='block_number', how='inner')
merged_df

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty,block_timestamp,real_outputs,real_outputs_parsed,n_addresses,primary_miner_result,primary_miner_address,primary_miner_ties
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-03 18:15:05+00:00,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,1,"(1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa, 1)",1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa,1
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:54:25+00:00,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,1,"(12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX, 1)",12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX,1
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:55:44+00:00,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,1,"(1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1, 1)",1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1,1
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:02:53+00:00,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,1,"(1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR, 1)",1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR,1
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:16:28+00:00,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,1,"(15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG, 1)",15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13,2023-10-06 12:36:45+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13,2023-10-06 12:45:49+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13,2023-10-06 12:50:15+00:00,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,2,"(1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY, 1)",1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY,1
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13,2023-10-06 13:37:00+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1,"(38XnPvu9PmonFU9WouPXUjYbW91wa5MerL, 1)",38XnPvu9PmonFU9WouPXUjYbW91wa5MerL,1


In [9]:
miner_block_counts = merged_df['primary_miner_address'].value_counts()
print(miner_block_counts)


primary_miner_address
1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY            81237
18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX            34093
1CK6KHY6MHgYvmRQ4PAafKYDrg1ejbH1cE            27261
14cZMQk89mRYQkDEj8Rn25AnGoBi5H6uer            26204
1CjPR7Z5ZSyWk6WtXvSFgkptmpoi4UM9BC            23083
                                              ...  
3EHpvbs5ar7DWiux1AQ5JMB2zTBE6wzX7d                1
bc1qrm09asqsxh2l7pjxjlltm53tx689dp2amzupzq        1
bc1q6fu02uvz2ghz3e3avqvqsqscr2pp4xhnuqfxpa        1
bc1q2za4ejga366sn288273pty8trasn5zs4y9hqg6        1
15h4MFgMs3yGiGfXA7fqhpsTWMkQ95EFBB                1
Name: count, Length: 197095, dtype: int64


In [10]:
# Step A: for every row, look up how many blocks its own address has mined in total
# .map() works like a dictionary lookup applied to an entire column at once,
# and importantly, unlike direct indexing, .map() quietly returns NaN for addresses
# it can't find (like our two None addresses), instead of crashing
merged_df['own_block_count'] = merged_df['primary_miner_address'].map(miner_block_counts)

# Step B: for every row, decide: keep the real address if count >= 10, otherwise 'long_tail'
merged_df['miner_group'] = np.where(merged_df['own_block_count'] >= 10, merged_df['primary_miner_address'], 'long_tail')

merged_df

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty,block_timestamp,real_outputs,real_outputs_parsed,n_addresses,primary_miner_result,primary_miner_address,primary_miner_ties,own_block_count,miner_group
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-03 18:15:05+00:00,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,1,"(1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa, 1)",1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa,1,1.0,long_tail
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:54:25+00:00,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,1,"(12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX, 1)",12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX,1,1.0,long_tail
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:55:44+00:00,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,1,"(1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1, 1)",1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1,1,1.0,long_tail
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:02:53+00:00,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,1,"(1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR, 1)",1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR,1,1.0,long_tail
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:16:28+00:00,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,1,"(15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG, 1)",15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG,1,1.0,long_tail
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13,2023-10-06 12:36:45+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1,14480.0,bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13,2023-10-06 12:45:49+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1,14480.0,bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13,2023-10-06 12:50:15+00:00,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,2,"(1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY, 1)",1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY,1,81237.0,1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13,2023-10-06 13:37:00+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1,"(38XnPvu9PmonFU9WouPXUjYbW91wa5MerL, 1)",38XnPvu9PmonFU9WouPXUjYbW91wa5MerL,1,15630.0,38XnPvu9PmonFU9WouPXUjYbW91wa5MerL


In [11]:
merged_df['miner_group'].value_counts()

miner_group
long_tail                             200987
1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY     81237
18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX     34093
1CK6KHY6MHgYvmRQ4PAafKYDrg1ejbH1cE     27261
14cZMQk89mRYQkDEj8Rn25AnGoBi5H6uer     26204
                                       ...  
1s2mUUUUGHzBsEy9p8CEUW82impM3fnav         10
1MVA4tCYxtCmfq5fz4LoiYTwWKaeB6LPEx        10
13WmMzyrJxfJAgWupbQGqCn9YeNQo2gJAW        10
12ybPxJQqJMFQK24Gszz8TdC1tJjWCm3Ch        10
1ZULUPooLEQfkrTgynLV4uHyMgQYx71ip         10
Name: count, Length: 976, dtype: int64

In [12]:
comparison = merged_df.groupby('miner_group').agg(
    n_blocks=('block_number', 'count'),
    avg_transactions=('transaction_count', 'mean'),
    avg_volume=('total_output_satoshis_excl_coinbase', 'mean'),
    avg_size=('size', 'mean'),
    avg_difficulty=('difficulty', 'mean'),
).reset_index()

comparison

,miner_group,n_blocks,avg_transactions,avg_volume,avg_size,avg_difficulty
0,112th1SmjAKALwtpsdXuYQ4Uu8e9rbwEXt,20,307.800000,4.934083e+11,1.454196e+05,1.364458e+09
1,1134jVdutTZpDfsPjLu4U1vE3UURdWeFui,15,1122.666667,1.408473e+12,1.050330e+06,2.686701e+12
2,115ZtZBrhso9PbHBUh44gf6ycxmoHHEFp1,516,276.662791,5.837774e+12,1.172129e+05,2.775592e+06
3,11wC5KcbgrWRBb43cwADdVrxgyF8mndVC,92,2248.413043,1.166347e+12,1.201610e+06,1.753234e+13
4,122Z63d9uRgYaAugxawaB79o12fRNYwFc8,125,905.792000,7.013965e+11,4.629104e+05,3.638500e+12
...,...,...,...,...,...,...
971,bc1qvfzssz36y4gxcg9gh234rzem9k0vrdlx4kq5sg,96,2958.635417,5.716535e+11,1.741142e+06,5.147298e+13
972,bc1qwlrsvgtn99rqp3fgaxq6f6jkgms80rnej0a8tc,71,2122.281690,1.071779e+12,1.323265e+06,2.210247e+13
973,bc1qx9t2l3pyny2spqpqlye8svce70nppwtaxwdrp4,4701,1915.912146,1.615549e+12,1.227302e+06,1.937132e+13
974,bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,14480,2590.779075,5.855246e+11,1.668432e+06,4.631282e+13


In [13]:
comparison.sort_values('n_blocks', ascending=False).reset_index()

,index,miner_group,n_blocks,avg_transactions,avg_volume,avg_size,avg_difficulty
0,975,long_tail,200987,53.730097,3.767628e+11,2.618385e+04,4.692112e+10
1,660,1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY,81237,1572.123860,1.223580e+12,9.029435e+05,1.131776e+13
2,250,18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX,34093,1918.849119,1.415265e+12,1.132617e+06,1.671824e+13
3,394,1CK6KHY6MHgYvmRQ4PAafKYDrg1ejbH1cE,27261,1740.016984,1.361256e+12,9.669289e+05,9.667326e+12
4,104,14cZMQk89mRYQkDEj8Rn25AnGoBi5H6uer,26204,384.621737,6.967644e+11,1.992497e+05,4.461016e+09
...,...,...,...,...,...,...,...
971,643,1JqyZ1ZyXGeuYVtW6Rdb3pUyr1Rzovjsai,10,1317.900000,1.528522e+12,9.985185e+05,6.114054e+11
972,653,1K5mkSvF6hCKktrXpe4aqL24kuYkNu1DrB,10,219.900000,1.624221e+11,1.776496e+05,3.466143e+10
973,179,16aCWzQW4EiDKuzqfFZDtZ7B8Jo5uNG2mt,10,781.900000,9.286592e+11,4.527701e+05,3.701776e+10
974,372,1BhJddEW2neo87ANjw1NKfXtXtZQpyLHAD,10,281.400000,1.892580e+11,2.249596e+05,1.175655e+10


### Comparing  (>= 10 Blocks)  and (< 10 Blocks) Miners

In [14]:
df1 = comparison.describe()
df1

,n_blocks,avg_transactions,avg_volume,avg_size,avg_difficulty
count,976.000000,976.000000,9.760000e+02,9.760000e+02,9.760000e+02
mean,830.849385,939.210486,1.033673e+12,5.224072e+05,4.277221e+12
std,7315.834144,795.373197,1.488880e+12,4.520810e+05,9.184461e+12
min,10.000000,3.933333,7.002614e+09,2.925067e+03,1.574164e+05
25%,15.000000,285.295674,4.583670e+11,1.419660e+05,1.874589e+08
50%,38.000000,560.971861,7.514152e+11,3.335523e+05,3.978064e+10
75%,168.000000,1762.025805,1.354585e+12,9.663208e+05,3.660821e+12
max,200987.000000,3190.382413,3.895685e+13,2.254906e+06,5.371538e+13


In [15]:
# describe without 'long_tail' row
df2 = comparison[comparison['miner_group'] != 'long_tail'].describe()
df2

,n_blocks,avg_transactions,avg_volume,avg_size,avg_difficulty
count,975.000000,975.000000,9.750000e+02,9.750000e+02,9.750000e+02
mean,625.561026,940.118671,1.034346e+12,5.229162e+05,4.281559e+12
std,3521.686866,795.274919,1.489495e+12,4.520332e+05,9.188173e+12
min,10.000000,3.933333,7.002614e+09,2.925067e+03,1.574164e+05
25%,15.000000,285.956342,4.585723e+11,1.420620e+05,1.854632e+08
50%,38.000000,561.181818,7.525184e+11,3.338283e+05,3.960367e+10
75%,167.500000,1763.088085,1.354779e+12,9.663877e+05,3.683142e+12
max,81237.000000,3190.382413,3.895685e+13,2.254906e+06,5.371538e+13


In [16]:
mismatches = df1.compare(df2)
mismatches

n_blocks               avg_transactions               \
                self         other             self        other   
count     976.000000    975.000000       976.000000   975.000000   
mean      830.849385    625.561026       939.210486   940.118671   
std      7315.834144   3521.686866       795.373197   795.274919   
25%              NaN           NaN       285.295674   285.956342   
50%              NaN           NaN       560.971861   561.181818   
75%       168.000000    167.500000      1762.025805  1763.088085   
max    200987.000000  81237.000000              NaN          NaN   

         avg_volume                     avg_size                 \
               self         other           self          other   
count  9.760000e+02  9.750000e+02     976.000000     975.000000   
mean   1.033673e+12  1.034346e+12  522407.223147  522916.170191   
std    1.488880e+12  1.489495e+12  452081.012329  452033.189714   
25%    4.583670e+11  4.585723e+11  141966.021429  142062.030005   
50%    7.514152e+11  7.525184e+11  333552.347222  333828.333333   
75%    1.354585e+12  1.354779e+12  966320.763594  966387.708493   
max             NaN           NaN            NaN            NaN   

      avg_difficulty                
                self         other  
count   9.760000e+02  9.750000e+02  
mean    4.277221e+12  4.281559e+12  
std     9.184461e+12  9.188173e+12  
25%     1.874589e+08  1.854632e+08  
50%     3.978064e+10  3.960367e+10  
75%     3.660821e+12  3.683142e+12  
max              NaN           NaN

In [17]:
# since long_tail statistics are mostly below the 25th percentile,
# check if it's mostly from the early era of blocks
merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'])

is_long_tail = merged_df['miner_group'] == 'long_tail'

print("long_tail blocks:")
print(merged_df.loc[is_long_tail, 'timestamp'].describe())

print("\nreal-miner blocks:")
print(merged_df.loc[~is_long_tail, 'timestamp'].describe())

# confirmed that the long_tail of <10 blocks miners are mostly from early era bitcoin
# there are still some low block miners in every year
# (may be caused by miners only using an address once and switching)

long_tail blocks:
count                              200987
mean     2011-02-21 08:53:09.961022+00:00
min             2009-01-03 18:15:05+00:00
25%             2010-04-11 16:37:43+00:00
50%             2011-01-01 11:42:28+00:00
75%      2011-10-30 05:34:15.500000+00:00
max             2023-10-05 07:29:38+00:00
Name: timestamp, dtype: object

real-miner blocks:
count                              609922
mean     2018-02-01 07:58:53.960064+00:00
min             2011-04-29 17:34:22+00:00
25%      2015-04-12 11:54:31.750000+00:00
50%      2018-01-21 19:24:24.500000+00:00
75%      2020-11-23 01:41:11.500000+00:00
max             2023-10-06 13:37:21+00:00
Name: timestamp, dtype: object


## Data Preparation, Model Training and Testing

### Step 0: Patch the one zero-volume block

In [18]:
# note: only 1 block had 0 in total_output_satoshis
# fill all 0 values with the median total_output_satoshis 
merged_df['total_output_satoshis'] = merged_df['total_output_satoshis'].replace(0, np.nan)
merged_df['total_output_satoshis'] = merged_df['total_output_satoshis'].fillna(merged_df['total_output_satoshis'].median())

### Step 1: The feature-building function

In [19]:
def build_miner_features(data, group_col='miner_group', drop_long_tail=False):
    df = data.copy()
    if drop_long_tail:
        df = df[df[group_col] != 'long_tail'].copy()

    # per-block ratio, needed later for the mean_of_ratios efficiency definition
    df['block_efficiency'] = df['transaction_count'] / df['total_output_satoshis']

    # the existing pipeline's fee approximation (kept for comparability)
    df['fee_proxy'] = df['transaction_count'] * df['total_output_satoshis']

    miner_df = df.groupby(group_col).agg(
        blocks_mined=('block_number', 'count'),
        avg_transactions=('transaction_count', 'mean'),
        avg_volume=('total_output_satoshis', 'mean'),
        avg_block_size=('size', 'mean'),
        difficulty=('difficulty', 'mean'),
        mean_of_ratios_efficiency=('block_efficiency', 'mean'),
        avg_fee_proxy=('fee_proxy', 'mean'),
        fee_volatility_proxy=('fee_proxy', 'std'),
        profitability_proxy=('fee_proxy', 'sum'),
        avg_fee_real=('total_fee_satoshis', 'mean'),
        fee_volatility_real=('total_fee_satoshis', 'std'),
        profitability_real=('total_fee_satoshis', 'sum'),
        last_block_time=('timestamp', 'max'),
    ).reset_index().rename(columns={group_col: 'miner_id'})

    # match the existing pipeline's profitability adjustment (sum / (blocks_mined + 1))
    miner_df['profitability_proxy'] = miner_df['profitability_proxy'] / (miner_df['blocks_mined'] + 1)
    miner_df['profitability_real'] = miner_df['profitability_real'] / (miner_df['blocks_mined'] + 1)

    # ratio_of_means efficiency: computed from the already-averaged columns, not per-block
    miner_df['ratio_of_means_efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)

    # neighbor_gap age: sort miners by their last-mined time, gap to whoever's right before them
    sorted_miners = miner_df.sort_values('last_block_time').reset_index(drop=True)
    sorted_miners['age'] = sorted_miners['last_block_time'].diff().dt.total_seconds()
    sorted_miners['age'] = sorted_miners['age'].fillna(sorted_miners['age'].median())
    miner_df = miner_df.merge(sorted_miners[['miner_id', 'age']], on='miner_id')

    return miner_df

### Step 2: Build both feature tables

In [20]:
bucket_features = build_miner_features(merged_df, drop_long_tail=False)
drop_features = build_miner_features(merged_df, drop_long_tail=True)

### Step 3: Add labels for both efficiency definitions

In [21]:
def add_label(miner_df, efficiency_col, label_col):
    median_eff = miner_df[efficiency_col].median()
    miner_df[label_col] = (miner_df[efficiency_col] > median_eff).astype(int)

for df_ in [bucket_features, drop_features]:
    add_label(df_, 'ratio_of_means_efficiency', 'label_rom')
    add_label(df_, 'mean_of_ratios_efficiency', 'label_mor')

### Step 4: The model-running function

In [22]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

def run_pipeline(miner_df, feature_columns, label_col, seed=42):
    X = miner_df[feature_columns]
    y = miner_df[label_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam',
        alpha=0.001, random_state=seed, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, max_iter=100, batch_size=16
    )
    model.fit(X_train_scaled, y_train)

    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    cv_mean = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()

    return test_acc, cv_mean

### Step 5: Run all 8 configurations and collect results

In [23]:
# test the model with full feature set
feature_sets = {
    'proxy_fee': ['blocks_mined', 'avg_transactions', 'avg_volume', 'avg_fee_proxy',
                  'fee_volatility_proxy', 'avg_block_size', 'difficulty', 'profitability_proxy', 'age'],
    'real_fee': ['blocks_mined', 'avg_transactions', 'avg_volume', 'avg_fee_real',
                 'fee_volatility_real', 'avg_block_size', 'difficulty', 'profitability_real', 'age'],
}
label_cols = {'ratio_of_means': 'label_rom', 'mean_of_ratios': 'label_mor'}
grouping_schemes = {'bucket': bucket_features, 'drop': drop_features}

results = []
for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets.items():
        for eff_name, label_col in label_cols.items():
            test_acc, cv_mean = run_pipeline(miner_df, feature_cols, label_col)
            results.append({
                'grouping': grouping_name, 'fee_measure': fee_name, 'efficiency': eff_name,
                'test_accuracy': test_acc, 'cv_mean': cv_mean, 'n_miners': len(miner_df)
            })

results_df = pd.DataFrame(results)
results_df

,grouping,fee_measure,efficiency,test_accuracy,cv_mean,n_miners
0,bucket,proxy_fee,ratio_of_means,0.943878,0.943590,976
1,bucket,proxy_fee,mean_of_ratios,0.882653,0.874359,976
2,bucket,real_fee,ratio_of_means,0.948980,0.953846,976
3,bucket,real_fee,mean_of_ratios,0.882653,0.882051,976
4,drop,proxy_fee,ratio_of_means,0.958974,0.960256,975
5,drop,proxy_fee,mean_of_ratios,0.887179,0.892308,975
6,drop,real_fee,ratio_of_means,0.948718,0.965385,975
7,drop,real_fee,mean_of_ratios,0.887179,0.885897,975


In [24]:
# test the model with reduced feature set
feature_sets_reduced = {
    'proxy_fee': ['blocks_mined', 'avg_fee_proxy', 'fee_volatility_proxy',
                  'avg_block_size', 'difficulty', 'profitability_proxy', 'age'],
    'real_fee': ['blocks_mined', 'avg_fee_real', 'fee_volatility_real',
                 'avg_block_size', 'difficulty', 'profitability_real', 'age'],
}

results_reduced = []
for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets_reduced.items():
        for eff_name, label_col in label_cols.items():
            test_acc, cv_mean = run_pipeline(miner_df, feature_cols, label_col)
            results_reduced.append({
                'grouping': grouping_name, 'fee_measure': fee_name, 'efficiency': eff_name,
                'test_accuracy': test_acc, 'cv_mean': cv_mean, 'n_miners': len(miner_df)
            })

results_reduced_df = pd.DataFrame(results_reduced)
results_reduced_df

,grouping,fee_measure,efficiency,test_accuracy,cv_mean,n_miners
0,bucket,proxy_fee,ratio_of_means,0.882653,0.861538,976
1,bucket,proxy_fee,mean_of_ratios,0.862245,0.841026,976
2,bucket,real_fee,ratio_of_means,0.780612,0.817949,976
3,bucket,real_fee,mean_of_ratios,0.816327,0.856410,976
4,drop,proxy_fee,ratio_of_means,0.887179,0.874359,975
5,drop,proxy_fee,mean_of_ratios,0.907692,0.856410,975
6,drop,real_fee,ratio_of_means,0.800000,0.839744,975
7,drop,real_fee,mean_of_ratios,0.835897,0.819231,975


## Analysing data skew

### Check Skew and Min Values

In [31]:
print(bucket_features[feature_sets_reduced['proxy_fee']].skew())
print(bucket_features[feature_sets_reduced['real_fee']].skew())

blocks_mined            22.703441
avg_fee_proxy            3.826786
fee_volatility_proxy     8.668703
avg_block_size           0.733748
difficulty               2.834933
profitability_proxy      3.514664
age                      4.921510
dtype: float64
blocks_mined           22.703441
avg_fee_real            3.609558
fee_volatility_real    15.977721
avg_block_size          0.733748
difficulty              2.834933
profitability_real      3.543607
age                     4.921510
dtype: float64


In [30]:
print(drop_features[feature_sets_reduced['proxy_fee']].skew())
print(drop_features[feature_sets_reduced['real_fee']].skew())

blocks_mined            15.035409
avg_fee_proxy            3.826464
fee_volatility_proxy     8.664621
avg_block_size           0.732712
difficulty               2.833091
profitability_proxy      3.514235
age                      4.919886
dtype: float64
blocks_mined           15.035409
avg_fee_real            3.607983
fee_volatility_real    15.969772
avg_block_size          0.732712
difficulty              2.833091
profitability_real      3.542021
age                     4.919886
dtype: float64


In [ ]:
print(bucket_features[feature_sets_reduced['proxy_fee']].min())
print(bucket_features[feature_sets_reduced['real_fee']].min())

blocks_mined            1.000000e+01
avg_fee_proxy           1.083209e+11
fee_volatility_proxy    1.241486e+11
avg_block_size          2.925067e+03
difficulty              1.574164e+05
profitability_proxy     9.929420e+10
age                     2.100000e+01
dtype: float64
blocks_mined               10.000000
avg_fee_real           517956.100000
fee_volatility_real    151609.532279
avg_block_size           2925.066667
difficulty             157416.401844
profitability_real     470869.181818
age                        21.000000
dtype: float64


In [ ]:
print(drop_features[feature_sets_reduced['proxy_fee']].min())
print(drop_features[feature_sets_reduced['real_fee']].min())

blocks_mined            1.000000e+01
avg_fee_proxy           1.083209e+11
fee_volatility_proxy    1.241486e+11
avg_block_size          2.925067e+03
difficulty              1.574164e+05
profitability_proxy     9.929420e+10
age                     2.100000e+01
dtype: float64
blocks_mined               10.000000
avg_fee_real           517956.100000
fee_volatility_real    151609.532279
avg_block_size           2925.066667
difficulty             157416.401844
profitability_real     470869.181818
age                        21.000000
dtype: float64


### Run Training and Tests with transformations applied to columns

#### log1p for all columns

In [ ]:
log_cols_map = {
    'proxy_fee': ['blocks_mined', 'avg_fee_proxy', 'fee_volatility_proxy', 'difficulty', 'profitability_proxy', 'age'],
    'real_fee': ['blocks_mined', 'avg_fee_real', 'fee_volatility_real', 'difficulty', 'profitability_real', 'age'],
}

print("=== Skew comparison: before vs after log1p ===\n")

for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets_reduced.items():
        cols_to_log = log_cols_map[fee_name]

        before = miner_df[cols_to_log].skew()

        df_log = miner_df.copy()
        df_log[cols_to_log] = np.log1p(df_log[cols_to_log])
        after = df_log[cols_to_log].skew()

        comparison = pd.DataFrame({'before_log1p': before, 'after_log1p': after})
        comparison['change'] = comparison['after_log1p'] - comparison['before_log1p']

        print(f"--- {grouping_name} / {fee_name} ---")
        print(comparison.round(3))
        print()

=== Skew comparison: before vs after log1p ===

--- bucket / proxy_fee ---
                      before_log1p  after_log1p  change
blocks_mined                22.703        1.212 -21.492
avg_fee_proxy                3.827       -1.819  -5.646
fee_volatility_proxy         8.669       -1.865 -10.534
difficulty                   2.835       -0.370  -3.205
profitability_proxy          3.515       -1.798  -5.313
age                          4.922       -0.781  -5.703

--- bucket / real_fee ---
                     before_log1p  after_log1p  change
blocks_mined               22.703        1.212 -21.492
avg_fee_real                3.610        0.359  -3.250
fee_volatility_real        15.978        0.497 -15.481
difficulty                  2.835       -0.370  -3.205
profitability_real          3.544        0.340  -3.203
age                         4.922       -0.781  -5.703

--- drop / proxy_fee ---
                      before_log1p  after_log1p  change
blocks_mined                15.035     

In [35]:
log_cols_map = {
    'proxy_fee': ['blocks_mined', 'avg_fee_proxy', 'fee_volatility_proxy', 'difficulty', 'profitability_proxy', 'age'],
    'real_fee': ['blocks_mined', 'avg_fee_real', 'fee_volatility_real', 'difficulty', 'profitability_real', 'age'],
}

results_log = []
for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets_reduced.items():
        for eff_name, label_col in label_cols.items():
            df_log = miner_df.copy()
            cols_to_log = log_cols_map[fee_name]
            df_log[cols_to_log] = np.log1p(df_log[cols_to_log])
            test_acc, cv_mean = run_pipeline(df_log, feature_cols, label_col)
            results_log.append({
                'grouping': grouping_name, 'fee_measure': fee_name, 'efficiency': eff_name,
                'test_accuracy': test_acc, 'cv_mean': cv_mean, 'n_miners': len(miner_df)
            })

results_log_df = pd.DataFrame(results_log)
results_log_df

,grouping,fee_measure,efficiency,test_accuracy,cv_mean,n_miners
0,bucket,proxy_fee,ratio_of_means,0.872449,0.841026,976
1,bucket,proxy_fee,mean_of_ratios,0.857143,0.866667,976
2,bucket,real_fee,ratio_of_means,0.795918,0.816667,976
3,bucket,real_fee,mean_of_ratios,0.852041,0.852564,976
4,drop,proxy_fee,ratio_of_means,0.866667,0.864103,975
5,drop,proxy_fee,mean_of_ratios,0.876923,0.884615,975
6,drop,real_fee,ratio_of_means,0.820513,0.820513,975
7,drop,real_fee,mean_of_ratios,0.882051,0.855128,975


#### check if there are more suitable transformations for age and blocks_mined

In [ ]:
test_transforms = pd.DataFrame({
    'age_raw_skew': [drop_features['age'].skew()],
    'age_sqrt_skew': [np.sqrt(drop_features['age']).skew()],
    'blocks_mined_log1p_skew': [np.log1p(drop_features['blocks_mined']).skew()],
    'blocks_mined_doublelog_skew': [np.log1p(np.log1p(drop_features['blocks_mined'])).skew()],
    'blocks_mined_sqrt_skew': [np.sqrt(drop_features['blocks_mined']).skew()],
})
test_transforms.round(3) 

,age_raw_skew,age_sqrt_skew,blocks_mined_log1p_skew,blocks_mined_doublelog_skew,blocks_mined_sqrt_skew
0,4.92,1.672,1.163,0.623,5.478


In [39]:
def transform_skewed_features(df, fee_name):
    df = df.copy()
    single_log_cols = {
        'proxy_fee': ['avg_fee_proxy', 'fee_volatility_proxy', 'difficulty', 'profitability_proxy', 'age'],
        'real_fee': ['avg_fee_real', 'fee_volatility_real', 'difficulty', 'profitability_real', 'age'],
    }[fee_name]

    df[single_log_cols] = np.log1p(df[single_log_cols])
    df['blocks_mined'] = np.log1p(np.log1p(df['blocks_mined']))
    return df

print("=== Skew check: updated transform (log1p everywhere, double log1p on blocks_mined) ===\n")

for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets_reduced.items():
        before = miner_df[feature_cols].skew()

        df_transformed = transform_skewed_features(miner_df, fee_name)
        after = df_transformed[feature_cols].skew()

        comparison = pd.DataFrame({'before': before, 'after': after})
        comparison['abs_after'] = comparison['after'].abs()

        print(f"--- {grouping_name} / {fee_name} ---")
        print(comparison.round(3))
        print()

=== Skew check: updated transform (log1p everywhere, double log1p on blocks_mined) ===

--- bucket / proxy_fee ---
                      before  after  abs_after
blocks_mined          22.703  0.638      0.638
avg_fee_proxy          3.827 -1.819      1.819
fee_volatility_proxy   8.669 -1.865      1.865
avg_block_size         0.734  0.734      0.734
difficulty             2.835 -0.370      0.370
profitability_proxy    3.515 -1.798      1.798
age                    4.922 -0.781      0.781

--- bucket / real_fee ---
                     before  after  abs_after
blocks_mined         22.703  0.638      0.638
avg_fee_real          3.610  0.359      0.359
fee_volatility_real  15.978  0.497      0.497
avg_block_size        0.734  0.734      0.734
difficulty            2.835 -0.370      0.370
profitability_real    3.544  0.340      0.340
age                   4.922 -0.781      0.781

--- drop / proxy_fee ---
                      before  after  abs_after
blocks_mined          15.035  0.623      

In [40]:
results_log_v2 = []
for grouping_name, miner_df in grouping_schemes.items():
    for fee_name, feature_cols in feature_sets_reduced.items():
        for eff_name, label_col in label_cols.items():
            df_transformed = transform_skewed_features(miner_df, fee_name)
            test_acc, cv_mean = run_pipeline(df_transformed, feature_cols, label_col)
            results_log_v2.append({
                'grouping': grouping_name, 'fee_measure': fee_name, 'efficiency': eff_name,
                'test_accuracy': test_acc, 'cv_mean': cv_mean, 'n_miners': len(miner_df)
            })

results_log_v2_df = pd.DataFrame(results_log_v2)
results_log_v2_df

,grouping,fee_measure,efficiency,test_accuracy,cv_mean,n_miners
0,bucket,proxy_fee,ratio_of_means,0.872449,0.866667,976
1,bucket,proxy_fee,mean_of_ratios,0.882653,0.882051,976
2,bucket,real_fee,ratio_of_means,0.831633,0.823077,976
3,bucket,real_fee,mean_of_ratios,0.857143,0.848718,976
4,drop,proxy_fee,ratio_of_means,0.841026,0.870513,975
5,drop,proxy_fee,mean_of_ratios,0.871795,0.885897,975
6,drop,real_fee,ratio_of_means,0.835897,0.817949,975
7,drop,real_fee,mean_of_ratios,0.887179,0.857692,975
